# Behavior Follow-Ups 1-4

Follow-up analyses for the gridworld action-commitment behavior experiment:

1. Permanent lock-in metric
2. Action-wise commitment curves
3. Strict pre-action analysis
4. Mismatched prompt/CoT control

This notebook is offline by default. It does not expand to the full 100 trajectories and does not call Together AI unless `RUN_TOGETHER_SCORING` is explicitly set to `True`.

## 0. Setup and Loading

In [ ]:

from pathlib import Path
import json
import math
import os
import re
import subprocess
import textwrap
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import scipy  # noqa: F401
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

# Opt-in only. Leave False for normal notebook runs.
RUN_TOGETHER_SCORING = False

PERMANENT_LOCK_MARGIN = 0.20
N_PROGRESS_BINS = 20
BUFFER_TOKENS = 5
RANDOM_SEED = 20260627
ACTIONS = ["UP", "DOWN", "LEFT", "RIGHT"]
Q_COLS = [f"q_{a}" for a in ACTIONS]
WARNINGS = []

START_CWD = Path.cwd()
workspace_fallback = Path("/Users/wws/Desktop/SPAR_code")
candidate_roots = [Path.cwd(), *Path.cwd().parents, workspace_fallback]
ROOT = None
for candidate in candidate_roots:
    if (candidate / "outputs").exists() and (candidate / "scripts").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    ROOT = Path.cwd().resolve()
    print(f"WARNING: could not locate repo root from {START_CWD}; using {ROOT}")
elif ROOT != Path.cwd().resolve():
    os.chdir(ROOT)
    print(f"Changed working directory from {START_CWD} to {ROOT}")

FIG_DIR = Path("figures/followups")
OUT_DIR = Path("outputs/followups")
for path in [FIG_DIR, OUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PREFIX_METRICS_PATH = Path("outputs/prefix_commitment_metrics.parquet")
STEP_SUMMARY_PATH = Path("outputs/step_commitment_summary.parquet")
MANIFOLD_PATH = Path("outputs/behavior_manifold.parquet")
PREFIXES_PATH = Path("outputs/prefixes.parquet")
PROMPT_ONLY_SCORES_PATH = Path("outputs/diagnostics/prompt_only_action_scores.parquet")
DIAG_MISMATCH_PREFIX_PATH = Path("outputs/diagnostics/mismatched_prompt_cot_prefixes.parquet")

FOLLOWUP_MISMATCH_PREFIX_PATH = OUT_DIR / "mismatched_prompt_cot_prefixes.parquet"
FOLLOWUP_MATCHED_PREFIX_PATH = OUT_DIR / "matched_prompt_cot_prefixes_subset.parquet"
FOLLOWUP_MISMATCH_SCORE_PATH = OUT_DIR / "mismatched_prompt_cot_scores.parquet"
FOLLOWUP_MATCHED_SCORE_PATH = OUT_DIR / "matched_prompt_cot_subset_scores.parquet"


def warn(msg):
    WARNINGS.append(str(msg))
    print(f"WARNING: {msg}")


def load_required(path):
    if not path.exists():
        raise FileNotFoundError(f"Required file is missing: {path}")
    print(f"Loading {path}")
    return pd.read_parquet(path)


def load_optional(path):
    if not path.exists():
        warn(f"Optional file missing; dependent section will skip if needed: {path}")
        return None
    print(f"Loading optional {path}")
    return pd.read_parquet(path)


def ensure_decision_id(df):
    if df is None:
        return None
    if "decision_id" not in df.columns and {"trajectory_id", "step_id"}.issubset(df.columns):
        df = df.copy()
        df["decision_id"] = df["trajectory_id"].astype(str) + "::" + df["step_id"].astype(str)
    return df


def q_for_action(df, action_col="agent_action"):
    out = []
    for _, row in df.iterrows():
        action = row.get(action_col)
        col = f"q_{action}"
        out.append(row[col] if col in row.index and pd.notna(row[col]) else np.nan)
    return pd.Series(out, index=df.index, dtype="float64")


def entropy_commitment_from_q(df):
    q = df[Q_COLS].astype(float).clip(lower=0).to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        entropy = -(np.where(q > 0, q * np.log(q), 0)).sum(axis=1)
    return 1 - entropy / np.log(4)


def recompute_prefix_metrics(df):
    df = df.copy()
    if not set(Q_COLS).issubset(df.columns):
        return df
    if "top_action" not in df.columns:
        df["top_action"] = df[Q_COLS].astype(float).idxmax(axis=1).str.replace("q_", "", regex=False)
    if "commitment_to_final" not in df.columns and "agent_action" in df.columns:
        df["commitment_to_final"] = q_for_action(df, "agent_action")
    if "entropy_commitment" not in df.columns:
        df["entropy_commitment"] = entropy_commitment_from_q(df)
    if "margin_commitment" not in df.columns:
        sorted_q = np.sort(df[Q_COLS].astype(float).to_numpy(), axis=1)
        df["margin_commitment"] = sorted_q[:, -1] - sorted_q[:, -2]
    if "is_top_action_final" not in df.columns and {"top_action", "agent_action"}.issubset(df.columns):
        df["is_top_action_final"] = df["top_action"] == df["agent_action"]
    return df


def final_prefix_rows(df):
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df.copy()
    return df.sort_values(["decision_id", "prefix_token_index"]).groupby("decision_id", as_index=False).tail(1)


def first_prefix_rows(df):
    if df is None or df.empty:
        return pd.DataFrame() if df is None else df.copy()
    return df.sort_values(["decision_id", "prefix_token_index"]).groupby("decision_id", as_index=False).head(1)


def standard_error(values):
    s = pd.Series(values).dropna()
    if len(s) <= 1:
        return np.nan
    return float(s.std(ddof=1) / np.sqrt(len(s)))


def save_fig(fig, filename):
    path = FIG_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {path}")
    return path


def safe_name(value, max_len=120):
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")
    return (text or "item")[:max_len]


def add_progress_bin(df, progress_col="progress", n_bins=N_PROGRESS_BINS):
    work = df.copy()
    edges = np.linspace(0, 1, n_bins + 1)
    work[progress_col] = work[progress_col].astype(float).clip(0, 1)
    work["progress_bin"] = pd.cut(work[progress_col], bins=edges, labels=False, include_lowest=True)
    work["progress_bin"] = work["progress_bin"].astype("Int64")
    return work, edges


def per_step_binned_curve(df, value_col, progress_col="progress", n_bins=N_PROGRESS_BINS):
    required = ["decision_id", progress_col, value_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        warn(f"Cannot build curve for {value_col}; missing {missing}")
        return pd.DataFrame(), pd.DataFrame()
    work = df[required].dropna().copy()
    if work.empty:
        return pd.DataFrame(), pd.DataFrame()
    work, edges = add_progress_bin(work, progress_col, n_bins)
    per_decision = work.groupby(["decision_id", "progress_bin"], observed=False)[value_col].mean().reset_index()
    summary = per_decision.groupby("progress_bin", observed=False)[value_col].agg(mean="mean", std="std", n_steps="count").reset_index()
    summary["sem"] = summary["std"] / np.sqrt(summary["n_steps"].replace(0, np.nan))
    counts = work.groupby("progress_bin", observed=False).agg(prefix_points=(value_col, "size"), decision_steps=("decision_id", "nunique")).reset_index()
    all_bins = pd.DataFrame({"progress_bin": np.arange(n_bins)})
    for frame in [summary, counts]:
        if not frame.empty:
            frame["progress_bin"] = frame["progress_bin"].astype(int)
    summary = all_bins.merge(summary, on="progress_bin", how="left")
    counts = all_bins.merge(counts, on="progress_bin", how="left").fillna({"prefix_points": 0, "decision_steps": 0})
    for frame in [summary, counts]:
        frame["bin_left"] = edges[frame["progress_bin"].to_numpy()]
        frame["bin_right"] = edges[frame["progress_bin"].to_numpy() + 1]
        frame["progress_mid"] = (frame["bin_left"] + frame["bin_right"]) / 2
    return summary, counts


def actionwise_binned_curve(df, value_col, progress_col="progress", n_bins=N_PROGRESS_BINS):
    required = ["decision_id", "agent_action", progress_col, value_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        warn(f"Cannot build action-wise curve for {value_col}; missing {missing}")
        return pd.DataFrame()
    work = df[required].dropna().copy()
    if work.empty:
        return pd.DataFrame()
    work, edges = add_progress_bin(work, progress_col, n_bins)
    per_decision = work.groupby(["agent_action", "decision_id", "progress_bin"], observed=False)[value_col].mean().reset_index()
    summary = per_decision.groupby(["agent_action", "progress_bin"], observed=False)[value_col].agg(mean="mean", std="std", n_steps="count").reset_index()
    summary["sem"] = summary["std"] / np.sqrt(summary["n_steps"].replace(0, np.nan))
    all_rows = pd.MultiIndex.from_product([ACTIONS, np.arange(n_bins)], names=["agent_action", "progress_bin"]).to_frame(index=False)
    summary = all_rows.merge(summary, on=["agent_action", "progress_bin"], how="left")
    summary["bin_left"] = edges[summary["progress_bin"].to_numpy()]
    summary["bin_right"] = edges[summary["progress_bin"].to_numpy() + 1]
    summary["progress_mid"] = (summary["bin_left"] + summary["bin_right"]) / 2
    summary["metric"] = value_col
    return summary


def plot_single_curve(summary, ylabel, title, filename, ylim=None):
    if summary.empty or summary["mean"].dropna().empty:
        warn(f"Skipping {filename}; no data")
        return None
    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    x = summary["progress_mid"].to_numpy(dtype=float)
    y = summary["mean"].to_numpy(dtype=float)
    sem = summary["sem"].fillna(0).to_numpy(dtype=float)
    ax.plot(x, y, marker="o", linewidth=2)
    ax.fill_between(x, y - sem, y + sem, alpha=0.2)
    ax.set_xlabel("progress")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(0, 1)
    if ylim is not None:
        ax.set_ylim(*ylim)
    return save_fig(fig, filename)


def plot_actionwise_curve(summary, ylabel, title, filename, ylim=None):
    if summary.empty or summary["mean"].dropna().empty:
        warn(f"Skipping {filename}; no data")
        return None
    fig, ax = plt.subplots(figsize=(8.0, 5.2))
    colors = {"UP": "#4C78A8", "DOWN": "#F58518", "LEFT": "#54A24B", "RIGHT": "#B279A2"}
    for action in ACTIONS:
        sub = summary[summary["agent_action"] == action]
        if sub["mean"].dropna().empty:
            continue
        x = sub["progress_mid"].to_numpy(dtype=float)
        y = sub["mean"].to_numpy(dtype=float)
        sem = sub["sem"].fillna(0).to_numpy(dtype=float)
        ax.plot(x, y, marker="o", linewidth=2, label=action, color=colors[action])
        ax.fill_between(x, y - sem, y + sem, alpha=0.12, color=colors[action])
    ax.set_xlabel("progress")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(0, 1)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.legend(title="agent_action")
    return save_fig(fig, filename)


def plot_counts(counts, title, filename):
    if counts.empty:
        warn(f"Skipping {filename}; no coverage data")
        return None
    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    ax.bar(counts["progress_mid"], counts["prefix_points"], width=0.04, alpha=0.75, label="prefix points")
    ax.set_xlabel("progress")
    ax.set_ylabel("prefix points")
    ax2 = ax.twinx()
    ax2.plot(counts["progress_mid"], counts["decision_steps"], color="black", marker="o", label="decision steps")
    ax2.set_ylabel("decision steps")
    ax.set_title(title)
    lines, labels = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines + lines2, labels + labels2, loc="best")
    ax.set_xlim(0, 1)
    return save_fig(fig, filename)


def categorical_scatter(df, x, y, color_col, title, filename):
    if df is None or df.empty or not {x, y, color_col}.issubset(df.columns):
        warn(f"Skipping {filename}; missing scatter columns")
        return None
    fig, ax = plt.subplots(figsize=(7.2, 5.6))
    for label, sub in df.groupby(color_col, dropna=False):
        ax.scatter(sub[x], sub[y], s=10, alpha=0.65, label=str(label))
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    ax.legend(markerscale=2, fontsize=8)
    return save_fig(fig, filename)


def command_to_text(args):
    if len(args) < 2:
        return " ".join(args)
    first = f"{args[0]} {args[1]}"
    entries = []
    i = 2
    while i < len(args):
        if str(args[i]).startswith("--") and i + 1 < len(args) and not str(args[i + 1]).startswith("--"):
            entries.append(f"  {args[i]} {args[i + 1]}")
            i += 2
        else:
            entries.append(f"  {args[i]}")
            i += 1
    continuation = " " + chr(92)
    lines = [first + continuation]
    for j, entry in enumerate(entries):
        suffix = continuation if j < len(entries) - 1 else ""
        lines.append(entry + suffix)
    return "\n".join(lines)

print(f"RUN_TOGETHER_SCORING = {RUN_TOGETHER_SCORING}")
print(f"scipy available: {SCIPY_AVAILABLE}")
print(f"root: {ROOT}")


In [ ]:

prefix_df = load_required(PREFIX_METRICS_PATH)
step_df = load_optional(STEP_SUMMARY_PATH)
manifold_df = load_optional(MANIFOLD_PATH)
prefixes_df = load_optional(PREFIXES_PATH)
prompt_only_file_df = load_optional(PROMPT_ONLY_SCORES_PATH)

prefix_df = ensure_decision_id(prefix_df)
step_df = ensure_decision_id(step_df)
manifold_df = ensure_decision_id(manifold_df)
prefixes_df = ensure_decision_id(prefixes_df)
prompt_only_file_df = ensure_decision_id(prompt_only_file_df)

# Enrich scored rows from the full prefix table where useful.
if prefixes_df is not None:
    enrich_cols = [
        "prefix_id", "decision_id", "trajectory_id", "source_file", "step_id", "prefix_token_index",
        "prompt_text", "analysis_prefix_text", "analysis_prefix_token_ids_json", "num_analysis_tokens",
        "first_action_mention_token_index", "first_final_action_mention_token_index",
        "is_before_first_action_mention", "is_before_first_final_action_mention",
    ]
    enrich_cols = [c for c in enrich_cols if c in prefixes_df.columns]
    if "prefix_id" in prefix_df.columns and "prefix_id" in enrich_cols:
        enrich = prefixes_df[enrich_cols].drop_duplicates("prefix_id")
        add_cols = [c for c in enrich.columns if c != "prefix_id" and c not in prefix_df.columns]
        if add_cols:
            prefix_df = prefix_df.merge(enrich[["prefix_id"] + add_cols], on="prefix_id", how="left")
            print(f"Enriched prefix_df with {add_cols}")
    elif {"decision_id", "prefix_token_index"}.issubset(prefix_df.columns) and {"decision_id", "prefix_token_index"}.issubset(enrich_cols):
        enrich = prefixes_df[enrich_cols].drop_duplicates(["decision_id", "prefix_token_index"])
        add_cols = [c for c in enrich.columns if c not in ["decision_id", "prefix_token_index"] and c not in prefix_df.columns]
        if add_cols:
            prefix_df = prefix_df.merge(enrich[["decision_id", "prefix_token_index"] + add_cols], on=["decision_id", "prefix_token_index"], how="left")
            print(f"Enriched prefix_df with {add_cols}")

prefix_df = recompute_prefix_metrics(prefix_df)
if step_df is not None:
    step_df = recompute_prefix_metrics(step_df) if set(Q_COLS).issubset(step_df.columns) else step_df
if manifold_df is not None:
    manifold_df = recompute_prefix_metrics(manifold_df) if set(Q_COLS).issubset(manifold_df.columns) else manifold_df

required_cols = [
    "decision_id", "trajectory_id", "step_id", "prefix_id", "prefix_token_index", "progress", "agent_action",
    "q_UP", "q_DOWN", "q_LEFT", "q_RIGHT", "top_action", "commitment_to_final", "entropy_commitment", "margin_commitment",
]
missing_required = [c for c in required_cols if c not in prefix_df.columns]
if missing_required:
    print("Missing required prefix-level columns:", missing_required)
else:
    print("All required prefix-level columns are present.")

prefix_counts = prefix_df.groupby("decision_id").size()
final_rows = final_prefix_rows(prefix_df)

dataset_summary = pd.DataFrame([
    {"metric": "prefix_points", "value": len(prefix_df)},
    {"metric": "decision_steps", "value": prefix_df["decision_id"].nunique()},
    {"metric": "trajectories", "value": prefix_df["trajectory_id"].nunique() if "trajectory_id" in prefix_df.columns else np.nan},
    {"metric": "prefixes_per_decision_min", "value": prefix_counts.min()},
    {"metric": "prefixes_per_decision_median", "value": prefix_counts.median()},
    {"metric": "prefixes_per_decision_mean", "value": prefix_counts.mean()},
    {"metric": "prefixes_per_decision_max", "value": prefix_counts.max()},
])
action_counts_decision = final_rows["agent_action"].value_counts().reindex(ACTIONS).fillna(0).astype(int)
action_counts_prefix = prefix_df["agent_action"].value_counts().reindex(ACTIONS).fillna(0).astype(int)
for action in ACTIONS:
    dataset_summary.loc[len(dataset_summary)] = {"metric": f"decision_count_{action}", "value": action_counts_decision[action]}
    dataset_summary.loc[len(dataset_summary)] = {"metric": f"prefix_count_{action}", "value": action_counts_prefix[action]}

dataset_summary.to_csv(OUT_DIR / "followup_dataset_summary.csv", index=False)

print("Prefix points:", len(prefix_df))
print("Decision steps:", prefix_df["decision_id"].nunique())
print("Trajectories:", prefix_df["trajectory_id"].nunique() if "trajectory_id" in prefix_df.columns else "NA")
print("Action counts by decision step")
display(action_counts_decision.rename("decision_steps"))
print("Action counts by prefix row")
display(action_counts_prefix.rename("prefix_rows"))
print("Prefixes per decision")
display(prefix_counts.describe())
display(dataset_summary)


## 1. Follow-Up 1 - Permanent Lock-In Metric

In [ ]:

def compute_permanent_lock_for_group(group, margin_threshold=PERMANENT_LOCK_MARGIN):
    group = group.sort_values("prefix_token_index").copy()
    agent_action = group["agent_action"].iloc[0]
    top_is_final = (group["top_action"] == agent_action).fillna(False)
    strong_final = (top_is_final & (group["margin_commitment"].astype(float) >= margin_threshold)).fillna(False)
    suffix_strong = strong_final.iloc[::-1].cummin().iloc[::-1].astype(bool)
    suffix_top = top_is_final.iloc[::-1].cummin().iloc[::-1].astype(bool)

    permanent_lock = group.loc[suffix_strong, "progress"].iloc[0] if suffix_strong.any() else np.nan
    permanent_top_lock = group.loc[suffix_top, "progress"].iloc[0] if suffix_top.any() else np.nan

    changes = (group["top_action"] != group["top_action"].shift()).fillna(False)
    changes.iloc[0] = False
    switch_count_total = int(changes.sum())

    final = group.tail(1).iloc[0]
    return {
        "decision_id": group["decision_id"].iloc[0],
        "trajectory_id": group["trajectory_id"].iloc[0] if "trajectory_id" in group.columns else None,
        "step_id": group["step_id"].iloc[0] if "step_id" in group.columns else None,
        "agent_action": agent_action,
        "permanent_lock_in_progress": permanent_lock,
        "permanent_top_action_lock_progress": permanent_top_lock,
        "switch_count_total": switch_count_total,
        "final_prefix_commitment_to_final": final.get("commitment_to_final"),
        "final_prefix_entropy_commitment": final.get("entropy_commitment"),
        "final_prefix_top_action": final.get("top_action"),
        "final_prefix_is_top_action_final": bool(final.get("top_action") == agent_action),
    }

permanent_rows = [compute_permanent_lock_for_group(g) for _, g in prefix_df.groupby("decision_id")]
permanent_lock_df = pd.DataFrame(permanent_rows)

if step_df is not None and {"decision_id", "lock_in_progress"}.issubset(step_df.columns):
    first_lock = step_df[["decision_id", "lock_in_progress"]].drop_duplicates("decision_id").rename(columns={"lock_in_progress": "first_lock_in_progress_existing"})
    permanent_lock_df = permanent_lock_df.merge(first_lock, on="decision_id", how="left")
else:
    permanent_lock_df["first_lock_in_progress_existing"] = np.nan
    warn("step_commitment_summary.lock_in_progress unavailable; first_lock_in_progress_existing set to NaN")

# Count switches after first/permanent locks.
switch_after_first = []
switch_after_permanent = []
for decision_id, group in prefix_df.groupby("decision_id"):
    group = group.sort_values("prefix_token_index").copy()
    changes = (group["top_action"] != group["top_action"].shift()).fillna(False)
    changes.iloc[0] = False
    row = permanent_lock_df.loc[permanent_lock_df["decision_id"] == decision_id].iloc[0]
    first_lock = row["first_lock_in_progress_existing"]
    permanent_lock = row["permanent_lock_in_progress"]
    switch_after_first.append(int(changes[group["progress"] > first_lock].sum()) if pd.notna(first_lock) else np.nan)
    switch_after_permanent.append(int(changes[group["progress"] > permanent_lock].sum()) if pd.notna(permanent_lock) else np.nan)
permanent_lock_df["switch_count_after_first_lock"] = switch_after_first
permanent_lock_df["switch_count_after_permanent_lock"] = switch_after_permanent

ordered_cols = [
    "decision_id", "trajectory_id", "step_id", "agent_action", "first_lock_in_progress_existing",
    "permanent_lock_in_progress", "permanent_top_action_lock_progress", "switch_count_total",
    "switch_count_after_first_lock", "switch_count_after_permanent_lock",
    "final_prefix_commitment_to_final", "final_prefix_entropy_commitment", "final_prefix_top_action", "final_prefix_is_top_action_final",
]
permanent_lock_df = permanent_lock_df[[c for c in ordered_cols if c in permanent_lock_df.columns]]
permanent_lock_df.to_parquet(OUT_DIR / "permanent_lockin_summary.parquet", index=False)
permanent_lock_df.to_csv(OUT_DIR / "permanent_lockin_summary.csv", index=False)

# Plots.
fig, ax = plt.subplots(figsize=(7.5, 5.0))
if permanent_lock_df["first_lock_in_progress_existing"].notna().any():
    ax.hist(permanent_lock_df["first_lock_in_progress_existing"].dropna(), bins=20, alpha=0.65, label="first lock-in")
ax.hist(permanent_lock_df["permanent_lock_in_progress"].dropna(), bins=20, alpha=0.65, label="permanent lock-in")
ax.set_xlabel("progress")
ax.set_ylabel("decision steps")
ax.set_title("First vs permanent lock-in progress")
ax.legend()
save_fig(fig, "first_vs_permanent_lockin_histogram.png")

fig, ax = plt.subplots(figsize=(7.5, 5.0))
ax.hist(permanent_lock_df["permanent_lock_in_progress"].dropna(), bins=20, alpha=0.85)
ax.set_xlabel("permanent lock-in progress")
ax.set_ylabel("decision steps")
ax.set_title("Permanent lock-in progress")
save_fig(fig, "permanent_lockin_progress_histogram.png")

fig, ax = plt.subplots(figsize=(6.0, 6.0))
scatter_data = permanent_lock_df.dropna(subset=["first_lock_in_progress_existing", "permanent_lock_in_progress"])
ax.scatter(scatter_data["first_lock_in_progress_existing"], scatter_data["permanent_lock_in_progress"], s=24, alpha=0.75)
ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1)
ax.set_xlabel("first lock-in progress")
ax.set_ylabel("permanent lock-in progress")
ax.set_title("First vs permanent lock-in")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
save_fig(fig, "first_vs_permanent_lockin_scatter.png")

fig, ax = plt.subplots(figsize=(7.5, 5.0))
ax.scatter(permanent_lock_df["switch_count_total"], permanent_lock_df["permanent_lock_in_progress"], s=24, alpha=0.75)
ax.set_xlabel("switch count total")
ax.set_ylabel("permanent lock-in progress")
ax.set_title("Switch count vs permanent lock-in")
save_fig(fig, "switch_count_vs_permanent_lockin.png")

fig, ax = plt.subplots(figsize=(7.5, 5.0))
ax.hist(permanent_lock_df["switch_count_after_first_lock"].dropna(), bins=range(0, int(permanent_lock_df["switch_count_after_first_lock"].dropna().max() if permanent_lock_df["switch_count_after_first_lock"].notna().any() else 0) + 3), alpha=0.85)
ax.set_xlabel("switch count after first lock")
ax.set_ylabel("decision steps")
ax.set_title("Switches after first lock-in")
save_fig(fig, "switch_count_after_first_lock_histogram.png")

example_cols = [c for c in ["decision_id", "agent_action", "first_lock_in_progress_existing", "permanent_lock_in_progress", "switch_count_after_first_lock", "switch_count_total", "final_prefix_commitment_to_final"] if c in permanent_lock_df.columns]
early_first_late_perm = permanent_lock_df[
    (permanent_lock_df["first_lock_in_progress_existing"] <= 0.25) &
    (permanent_lock_df["permanent_lock_in_progress"] >= 0.60)
].sort_values(["permanent_lock_in_progress", "switch_count_after_first_lock"], ascending=[False, False])[example_cols].head(30)
high_switch_after_first = permanent_lock_df.sort_values("switch_count_after_first_lock", ascending=False)[example_cols].head(30)
no_permanent_lock = permanent_lock_df[permanent_lock_df["permanent_lock_in_progress"].isna()][example_cols].head(30)
early_first_late_perm.to_csv(OUT_DIR / "early_first_late_permanent_lock_examples.csv", index=False)
high_switch_after_first.to_csv(OUT_DIR / "high_switch_after_first_lock_examples.csv", index=False)
no_permanent_lock.to_csv(OUT_DIR / "no_permanent_lock_examples.csv", index=False)

permanent_lock_report = {
    "median_first_lock_in_progress": permanent_lock_df["first_lock_in_progress_existing"].median(),
    "median_permanent_lock_in_progress": permanent_lock_df["permanent_lock_in_progress"].median(),
    "fraction_with_permanent_lock_in": permanent_lock_df["permanent_lock_in_progress"].notna().mean(),
    "mean_switch_count_after_first_lock": permanent_lock_df["switch_count_after_first_lock"].mean(),
}
display(pd.DataFrame([permanent_lock_report]))
display(permanent_lock_df.head())


## 2. Follow-Up 2 - Action-Wise Commitment Curves

In [ ]:

def build_prompt_only_baseline(prefix_df, prompt_only_file_df=None):
    source = None
    prompt_only = None
    if prompt_only_file_df is not None and set(Q_COLS + ["agent_action", "decision_id"]).issubset(prompt_only_file_df.columns):
        prompt_only = recompute_prefix_metrics(prompt_only_file_df)
        source = str(PROMPT_ONLY_SCORES_PATH)
    elif "prefix_token_index" in prefix_df.columns and (prefix_df["prefix_token_index"] == 0).any():
        prompt_only = first_prefix_rows(prefix_df[prefix_df["prefix_token_index"] == 0].copy())
        source = "existing prefix_token_index == 0 rows"
    else:
        warn("Prompt-only baseline unavailable: no prompt-only score file and no prefix_token_index == 0 rows")
        return None, None
    prompt_only = prompt_only.copy()
    if "prompt_only_commitment_to_final" not in prompt_only.columns:
        prompt_only["prompt_only_commitment_to_final"] = prompt_only["commitment_to_final"] if "commitment_to_final" in prompt_only.columns else q_for_action(prompt_only, "agent_action")
    if "prompt_only_entropy_commitment" not in prompt_only.columns:
        prompt_only["prompt_only_entropy_commitment"] = prompt_only["entropy_commitment"] if "entropy_commitment" in prompt_only.columns else entropy_commitment_from_q(prompt_only)
    return prompt_only[["decision_id", "agent_action", "prompt_only_commitment_to_final", "prompt_only_entropy_commitment"]].drop_duplicates("decision_id"), source

prompt_only_df, prompt_only_source = build_prompt_only_baseline(prefix_df, prompt_only_file_df)
action_prefix_df = prefix_df.copy()
if prompt_only_df is not None:
    action_prefix_df = action_prefix_df.merge(prompt_only_df[["decision_id", "prompt_only_commitment_to_final", "prompt_only_entropy_commitment"]], on="decision_id", how="left")
    action_prefix_df["delta_commitment_to_final"] = action_prefix_df["commitment_to_final"] - action_prefix_df["prompt_only_commitment_to_final"]
    action_prefix_df["delta_entropy_commitment"] = action_prefix_df["entropy_commitment"] - action_prefix_df["prompt_only_entropy_commitment"]
    print(f"Prompt-only baseline source: {prompt_only_source}")
else:
    print("No prompt-only baseline available for delta plots.")

curve_specs = [
    ("commitment_to_final", "commitment to final action", "Action-wise commitment to final action", "actionwise_commitment_to_final_over_progress.png", (0, 1)),
    ("entropy_commitment", "entropy commitment", "Action-wise entropy commitment", "actionwise_entropy_commitment_over_progress.png", (0, 1)),
    ("margin_commitment", "margin commitment", "Action-wise margin commitment", "actionwise_margin_commitment_over_progress.png", (0, 1)),
    ("is_top_action_final", "top-action accuracy", "Action-wise top-action accuracy", "actionwise_top_action_accuracy_over_progress.png", (0, 1)),
]
if "delta_commitment_to_final" in action_prefix_df.columns:
    curve_specs.insert(1, ("delta_commitment_to_final", "delta commitment vs prompt-only", "Action-wise commitment gain over prompt-only", "actionwise_delta_commitment_over_progress.png", None))

curve_tables = []
for metric, ylabel, title, filename, ylim in curve_specs:
    summary = actionwise_binned_curve(action_prefix_df, metric, n_bins=N_PROGRESS_BINS)
    if not summary.empty:
        curve_tables.append(summary)
    plot_actionwise_curve(summary, ylabel, title, filename, ylim=ylim)

actionwise_curve_table = pd.concat(curve_tables, ignore_index=True) if curve_tables else pd.DataFrame()
actionwise_curve_table.to_csv(OUT_DIR / "actionwise_progress_curves.csv", index=False)

# Prompt-only and final-minus-prompt bars.
final_action_rows = final_prefix_rows(action_prefix_df)
if prompt_only_df is not None:
    prompt_action = prompt_only_df.groupby("agent_action")["prompt_only_commitment_to_final"].mean().reindex(ACTIONS)
    fig, ax = plt.subplots(figsize=(7.0, 4.8))
    ax.bar(prompt_action.index, prompt_action.values, alpha=0.85)
    ax.set_xlabel("agent_action")
    ax.set_ylabel("mean prompt-only commitment")
    ax.set_ylim(0, 1)
    ax.set_title("Prompt-only commitment by final action")
    save_fig(fig, "actionwise_prompt_only_commitment.png")

    gain = final_action_rows.groupby("agent_action").apply(lambda g: (g["commitment_to_final"] - g["prompt_only_commitment_to_final"]).mean(), include_groups=False).reindex(ACTIONS)
    fig, ax = plt.subplots(figsize=(7.0, 4.8))
    ax.bar(gain.index, gain.values, alpha=0.85)
    ax.set_xlabel("agent_action")
    ax.set_ylabel("final minus prompt commitment")
    ax.set_title("Final-minus-prompt commitment gain by final action")
    save_fig(fig, "actionwise_final_minus_prompt_gain.png")
else:
    warn("Skipping prompt-only action-wise bar plots; prompt-only baseline unavailable")

if "num_analysis_tokens" in final_action_rows.columns:
    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    data = [final_action_rows.loc[final_action_rows["agent_action"] == a, "num_analysis_tokens"].dropna().values for a in ACTIONS]
    ax.boxplot(data, labels=ACTIONS, showfliers=False)
    ax.set_xlabel("agent_action")
    ax.set_ylabel("num analysis tokens")
    ax.set_title("CoT length by final action")
    save_fig(fig, "actionwise_cot_length_boxplot.png")

first_rows = first_prefix_rows(action_prefix_df)
if {"first_action_mention_token_index", "num_analysis_tokens"}.issubset(first_rows.columns):
    first_rows["first_action_mention_progress"] = first_rows["first_action_mention_token_index"] / first_rows["num_analysis_tokens"].replace({0: np.nan})
    fig, ax = plt.subplots(figsize=(7.0, 4.8))
    vals = first_rows.groupby("agent_action")["first_action_mention_progress"].mean().reindex(ACTIONS)
    ax.bar(vals.index, vals.values, alpha=0.85)
    ax.set_xlabel("agent_action")
    ax.set_ylabel("mean first action mention progress")
    ax.set_ylim(0, 1)
    ax.set_title("First action mention progress by final action")
    save_fig(fig, "actionwise_first_action_mention_progress.png")
else:
    first_rows["first_action_mention_progress"] = np.nan
    warn("Skipping first action mention progress plot; columns unavailable")

if {"first_final_action_mention_token_index", "num_analysis_tokens"}.issubset(first_rows.columns):
    first_rows["first_final_action_mention_progress"] = first_rows["first_final_action_mention_token_index"] / first_rows["num_analysis_tokens"].replace({0: np.nan})
    fig, ax = plt.subplots(figsize=(7.0, 4.8))
    vals = first_rows.groupby("agent_action")["first_final_action_mention_progress"].mean().reindex(ACTIONS)
    ax.bar(vals.index, vals.values, alpha=0.85)
    ax.set_xlabel("agent_action")
    ax.set_ylabel("mean first final-action mention progress")
    ax.set_ylim(0, 1)
    ax.set_title("First final-action mention progress by final action")
    save_fig(fig, "actionwise_first_final_action_mention_progress.png")
else:
    first_rows["first_final_action_mention_progress"] = np.nan
    warn("Skipping first final-action mention progress plot; columns unavailable")

summary_parts = []
summary = pd.DataFrame(index=ACTIONS)
summary["n_decisions"] = final_action_rows.groupby("agent_action")["decision_id"].nunique().reindex(ACTIONS)
summary["n_prefixes"] = action_prefix_df.groupby("agent_action")["decision_id"].size().reindex(ACTIONS)
if "num_analysis_tokens" in final_action_rows.columns:
    summary["mean_num_analysis_tokens"] = final_action_rows.groupby("agent_action")["num_analysis_tokens"].mean().reindex(ACTIONS)
    summary["median_num_analysis_tokens"] = final_action_rows.groupby("agent_action")["num_analysis_tokens"].median().reindex(ACTIONS)
if prompt_only_df is not None:
    summary["mean_prompt_only_commitment_to_final"] = prompt_only_df.groupby("agent_action")["prompt_only_commitment_to_final"].mean().reindex(ACTIONS)
summary["mean_final_prefix_commitment_to_final"] = final_action_rows.groupby("agent_action")["commitment_to_final"].mean().reindex(ACTIONS)
if prompt_only_df is not None:
    summary["mean_final_minus_prompt_commitment"] = final_action_rows.groupby("agent_action").apply(lambda g: (g["commitment_to_final"] - g["prompt_only_commitment_to_final"]).mean(), include_groups=False).reindex(ACTIONS)
summary["mean_first_action_mention_progress"] = first_rows.groupby("agent_action")["first_action_mention_progress"].mean().reindex(ACTIONS)
summary["mean_first_final_action_mention_progress"] = first_rows.groupby("agent_action")["first_final_action_mention_progress"].mean().reindex(ACTIONS)
if "permanent_lock_df" in globals() and not permanent_lock_df.empty:
    summary["median_permanent_lock_in_progress"] = permanent_lock_df.groupby("agent_action")["permanent_lock_in_progress"].median().reindex(ACTIONS)
    summary["mean_switch_count_total"] = permanent_lock_df.groupby("agent_action")["switch_count_total"].mean().reindex(ACTIONS)
actionwise_summary = summary.reset_index().rename(columns={"index": "agent_action"})
actionwise_summary.to_csv(OUT_DIR / "actionwise_summary.csv", index=False)

# Gain table for the final report.
action_gain_rows = []
for action in ACTIONS:
    sub = actionwise_curve_table[(actionwise_curve_table["agent_action"] == action) & (actionwise_curve_table["metric"] == "commitment_to_final")].dropna(subset=["mean"])
    if sub.empty:
        continue
    action_gain_rows.append({"agent_action": action, "curve_gain": sub["mean"].iloc[-1] - sub["mean"].iloc[0], "start_commitment": sub["mean"].iloc[0], "end_commitment": sub["mean"].iloc[-1]})
action_gain_table = pd.DataFrame(action_gain_rows)

display(actionwise_summary)
display(action_gain_table)


## 3. Follow-Up 3 - Strict Pre-Action Analysis

In [ ]:

strict_curve_frames = []
strict_coverage_frames = []
strict_pre_action_summary_rows = []
strict_available = {"first_action_mention_token_index", "first_final_action_mention_token_index", "prefix_token_index"}.issubset(action_prefix_df.columns)

if not strict_available:
    warn("Strict pre-action analysis skipped: first action/final-action mention token indices are unavailable")
    strict_work = pd.DataFrame()
else:
    strict_work = action_prefix_df.copy()
    strict_work["first_action_mention_token_index"] = pd.to_numeric(strict_work["first_action_mention_token_index"], errors="coerce")
    strict_work["first_final_action_mention_token_index"] = pd.to_numeric(strict_work["first_final_action_mention_token_index"], errors="coerce")
    strict_work["prefix_token_index"] = pd.to_numeric(strict_work["prefix_token_index"], errors="coerce")
    strict_work["strict_before_any_action_mention"] = strict_work["prefix_token_index"] < strict_work["first_action_mention_token_index"]
    strict_work["strict_before_final_action_mention"] = strict_work["prefix_token_index"] < strict_work["first_final_action_mention_token_index"]
    strict_work["strict_before_any_action_mention_buffered"] = strict_work["prefix_token_index"] < (strict_work["first_action_mention_token_index"] - BUFFER_TOKENS)
    strict_work["strict_before_final_action_mention_buffered"] = strict_work["prefix_token_index"] < (strict_work["first_final_action_mention_token_index"] - BUFFER_TOKENS)
    strict_work["progress_to_first_action_mention"] = strict_work["prefix_token_index"] / strict_work["first_action_mention_token_index"].replace({0: np.nan})
    strict_work["progress_to_first_final_action_mention"] = strict_work["prefix_token_index"] / strict_work["first_final_action_mention_token_index"].replace({0: np.nan})
    strict_work["progress_to_first_action_mention"] = strict_work["progress_to_first_action_mention"].clip(0, 1)
    strict_work["progress_to_first_final_action_mention"] = strict_work["progress_to_first_final_action_mention"].clip(0, 1)

    def add_curve(subset_name, mask, metric, progress_col):
        sub = strict_work[mask].dropna(subset=[progress_col]).copy()
        summary, counts = per_step_binned_curve(sub, metric, progress_col=progress_col, n_bins=N_PROGRESS_BINS)
        if not summary.empty:
            summary = summary.assign(subset=subset_name, metric=metric, progress_definition=progress_col)
            strict_curve_frames.append(summary)
        if not counts.empty:
            counts = counts.assign(subset=subset_name, progress_definition=progress_col)
            total_decisions = strict_work["decision_id"].nunique()
            counts["decision_fraction"] = counts["decision_steps"] / total_decisions if total_decisions else np.nan
            strict_coverage_frames.append(counts)
        return summary, counts

    any_mask = strict_work["strict_before_any_action_mention"].fillna(False)
    final_mask = strict_work["strict_before_final_action_mention"].fillna(False)
    any_buf_mask = strict_work["strict_before_any_action_mention_buffered"].fillna(False)
    final_buf_mask = strict_work["strict_before_final_action_mention_buffered"].fillna(False)

    any_full_commit, _ = add_curve("strict_before_any_action_mention", any_mask, "commitment_to_final", "progress")
    final_full_commit, _ = add_curve("strict_before_final_action_mention", final_mask, "commitment_to_final", "progress")
    any_norm_commit, any_norm_counts = add_curve("strict_before_any_action_mention", any_mask, "commitment_to_final", "progress_to_first_action_mention")
    final_norm_commit, final_norm_counts = add_curve("strict_before_final_action_mention", final_mask, "commitment_to_final", "progress_to_first_final_action_mention")
    any_norm_entropy, _ = add_curve("strict_before_any_action_mention", any_mask, "entropy_commitment", "progress_to_first_action_mention")
    final_norm_entropy, _ = add_curve("strict_before_final_action_mention", final_mask, "entropy_commitment", "progress_to_first_final_action_mention")
    any_buf_commit, _ = add_curve("strict_before_any_action_mention_buffered", any_buf_mask, "commitment_to_final", "progress_to_first_action_mention")
    final_buf_commit, _ = add_curve("strict_before_final_action_mention_buffered", final_buf_mask, "commitment_to_final", "progress_to_first_final_action_mention")
    any_buf_entropy, _ = add_curve("strict_before_any_action_mention_buffered", any_buf_mask, "entropy_commitment", "progress_to_first_action_mention")
    final_buf_entropy, _ = add_curve("strict_before_final_action_mention_buffered", final_buf_mask, "entropy_commitment", "progress_to_first_final_action_mention")

    plot_single_curve(any_full_commit, "commitment to final action", "Strict commitment before any action mention - full CoT progress", "strict_commitment_before_any_action_full_progress.png", ylim=(0, 1))
    plot_single_curve(final_full_commit, "commitment to final action", "Strict commitment before final-action mention - full CoT progress", "strict_commitment_before_final_action_full_progress.png", ylim=(0, 1))
    plot_single_curve(any_norm_commit, "commitment to final action", "Strict commitment normalized to first action mention", "strict_commitment_to_first_action_mention_progress.png", ylim=(0, 1))
    plot_single_curve(final_norm_commit, "commitment to final action", "Strict commitment normalized to first final-action mention", "strict_commitment_to_first_final_action_mention_progress.png", ylim=(0, 1))
    plot_single_curve(any_norm_entropy, "entropy commitment", "Strict entropy normalized to first action mention", "strict_entropy_to_first_action_mention_progress.png", ylim=(0, 1))
    plot_single_curve(final_norm_entropy, "entropy commitment", "Strict entropy normalized to first final-action mention", "strict_entropy_to_first_final_action_mention_progress.png", ylim=(0, 1))
    if "delta_commitment_to_final" in strict_work.columns:
        any_norm_delta, _ = add_curve("strict_before_any_action_mention", any_mask, "delta_commitment_to_final", "progress_to_first_action_mention")
        plot_single_curve(any_norm_delta, "delta commitment vs prompt-only", "Strict delta commitment normalized to first action mention", "strict_delta_commitment_to_first_action_mention_progress.png")
    else:
        warn("Skipping strict delta commitment plot; prompt-only baseline unavailable")

    plot_counts(any_norm_counts, "Strict coverage before any action mention", "strict_coverage_before_any_action_mention.png")
    plot_counts(final_norm_counts, "Strict coverage before final-action mention", "strict_coverage_before_final_action_mention.png")

    # PCA plots with strict masks if manifold is available.
    if manifold_df is not None and {"beh_pca1", "beh_pca2", "prefix_id"}.issubset(manifold_df.columns) and "prefix_id" in strict_work.columns:
        strict_flags = strict_work[["prefix_id", "strict_before_any_action_mention", "strict_before_final_action_mention"]].drop_duplicates("prefix_id")
        manifold_strict = manifold_df.merge(strict_flags, on="prefix_id", how="left")
        categorical_scatter(manifold_strict[manifold_strict["strict_before_any_action_mention"].fillna(False)], "beh_pca1", "beh_pca2", "agent_action", "PCA strict before any action mention by final action", "pca_strict_before_any_action_mention_by_agent_action.png")
        categorical_scatter(manifold_strict[manifold_strict["strict_before_final_action_mention"].fillna(False)], "beh_pca1", "beh_pca2", "agent_action", "PCA strict before final-action mention by final action", "pca_strict_before_final_action_mention_by_agent_action.png")
    else:
        warn("Skipping strict PCA plots; behavior_manifold or prefix_id/PCA columns unavailable")

    strict_curve_table = pd.concat(strict_curve_frames, ignore_index=True) if strict_curve_frames else pd.DataFrame()
    strict_coverage_table = pd.concat(strict_coverage_frames, ignore_index=True) if strict_coverage_frames else pd.DataFrame()
    strict_curve_table.to_csv(OUT_DIR / "strict_pre_action_curve_table.csv", index=False)
    strict_coverage_table.to_csv(OUT_DIR / "strict_pre_action_coverage_table.csv", index=False)

    def summarize_curve(curve, subset_name, metric, progress_definition):
        valid = curve.dropna(subset=["mean"]).copy()
        if valid.empty:
            return
        start = valid.iloc[0]
        mid = valid.iloc[(valid["progress_mid"] - 0.5).abs().idxmin()] if not valid.empty else None
        last = valid.iloc[-1]
        strict_pre_action_summary_rows.append({
            "subset": subset_name,
            "metric": metric,
            "progress_definition": progress_definition,
            "start_mean": start["mean"],
            "middle_mean": mid["mean"],
            "last_strict_bin_mean": last["mean"],
            "start_n_steps": start["n_steps"],
            "middle_n_steps": mid["n_steps"],
            "last_n_steps": last["n_steps"],
        })

    for curve_name, curve, metric, progress_def in [
        ("strict_before_any_action_mention", any_norm_commit, "commitment_to_final", "progress_to_first_action_mention"),
        ("strict_before_final_action_mention", final_norm_commit, "commitment_to_final", "progress_to_first_final_action_mention"),
        ("strict_before_any_action_mention", any_norm_entropy, "entropy_commitment", "progress_to_first_action_mention"),
        ("strict_before_final_action_mention", final_norm_entropy, "entropy_commitment", "progress_to_first_final_action_mention"),
        ("strict_before_any_action_mention_buffered", any_buf_commit, "commitment_to_final", "progress_to_first_action_mention"),
        ("strict_before_final_action_mention_buffered", final_buf_commit, "commitment_to_final", "progress_to_first_final_action_mention"),
        ("strict_before_any_action_mention_buffered", any_buf_entropy, "entropy_commitment", "progress_to_first_action_mention"),
        ("strict_before_final_action_mention_buffered", final_buf_entropy, "entropy_commitment", "progress_to_first_final_action_mention"),
    ]:
        summarize_curve(curve, curve_name, metric, progress_def)

    strict_pre_action_summary = pd.DataFrame(strict_pre_action_summary_rows)
    strict_pre_action_summary.to_csv(OUT_DIR / "strict_pre_action_summary.csv", index=False)
    display(strict_pre_action_summary)
    display(strict_coverage_table.head())


## 4. Follow-Up 4 - Mismatched Prompt/CoT Control

In [ ]:

def select_control_rows(source_df):
    selected = []
    for decision_id, group in source_df.groupby("decision_id"):
        group = group.sort_values("progress").copy()
        targets = [(0.0, "earliest_or_prompt"), (0.5, "mid"), (0.9, "late")]
        for target, label in targets:
            idx = (group["progress"].astype(float) - target).abs().idxmin()
            row = group.loc[idx].copy()
            row["control_position"] = label
            selected.append(row)
    return pd.DataFrame(selected).reset_index(drop=True) if selected else pd.DataFrame()


def normalize_existing_mismatch(df):
    df = ensure_decision_id(df).copy()
    if "prompt_source_decision_id" not in df.columns:
        df["prompt_source_decision_id"] = df.get("decision_id")
    if "cot_source_decision_id" not in df.columns:
        df["cot_source_decision_id"] = df.get("mismatch_source_decision_id", df.get("decision_id"))
    if "prompt_agent_action" not in df.columns:
        df["prompt_agent_action"] = df.get("agent_action")
    if "cot_agent_action" not in df.columns:
        df["cot_agent_action"] = df.get("mismatch_source_agent_action", df.get("cot_source_action", df.get("agent_action")))
    if "cot_source_action" not in df.columns:
        df["cot_source_action"] = df["cot_agent_action"]
    if "mismatch_type" not in df.columns:
        df["mismatch_type"] = "prompt_A_cot_B_different_action"
    df["agent_action"] = df["prompt_agent_action"]
    return df


def build_mismatch_and_matched(prefixes_source):
    required = {"decision_id", "trajectory_id", "step_id", "prompt_text", "analysis_prefix_text", "agent_action", "progress"}
    if prefixes_source is None or not required.issubset(prefixes_source.columns):
        warn(f"Cannot build mismatched prompt/CoT prefixes; outputs/prefixes.parquet missing required columns: {sorted(required)}")
        return pd.DataFrame(), pd.DataFrame()
    source = prefixes_source[prefixes_source["decision_id"].isin(prefix_df["decision_id"].unique())].copy()
    if source.empty:
        warn("No overlap between current scored decisions and outputs/prefixes.parquet")
        return pd.DataFrame(), pd.DataFrame()
    selected = select_control_rows(source)
    rng = np.random.default_rng(RANDOM_SEED)
    mismatch_rows = []
    matched_rows = []
    for _, row in selected.iterrows():
        same_position = selected[selected["control_position"] == row["control_position"]]
        donor_pool = same_position[same_position["agent_action"] != row["agent_action"]]
        if donor_pool.empty:
            donor_pool = selected[selected["agent_action"] != row["agent_action"]]
        if donor_pool.empty:
            donor_pool = selected[selected["decision_id"] != row["decision_id"]]
        if donor_pool.empty:
            continue
        donor = donor_pool.iloc[int(rng.integers(0, len(donor_pool)))]

        base_meta = {
            "trajectory_id": row.get("trajectory_id"),
            "source_file": row.get("source_file"),
            "step_id": row.get("step_id"),
            "prompt_source_decision_id": row.get("decision_id"),
            "prompt_agent_action": row.get("agent_action"),
            "control_position": row.get("control_position"),
        }
        mismatch = dict(base_meta)
        mismatch.update({
            "prefix_id": f"{row['decision_id']}::mismatch::{row['control_position']}",
            "decision_id": f"{row['decision_id']}::mismatch::{row['control_position']}",
            "cot_source_decision_id": donor.get("decision_id"),
            "cot_agent_action": donor.get("agent_action"),
            "agent_action": row.get("agent_action"),
            "cot_source_action": donor.get("agent_action"),
            "prompt_text": row.get("prompt_text"),
            "analysis_prefix_text": donor.get("analysis_prefix_text"),
            "analysis_prefix_token_ids_json": donor.get("analysis_prefix_token_ids_json"),
            "prefix_token_index": donor.get("prefix_token_index"),
            "num_analysis_tokens": donor.get("num_analysis_tokens"),
            "progress": donor.get("progress"),
            "mismatch_type": "prompt_A_cot_B_different_action",
        })
        mismatch_rows.append(mismatch)

        matched = dict(base_meta)
        matched.update({
            "prefix_id": f"{row['decision_id']}::matched::{row['control_position']}",
            "decision_id": f"{row['decision_id']}::matched::{row['control_position']}",
            "cot_source_decision_id": row.get("decision_id"),
            "cot_agent_action": row.get("agent_action"),
            "agent_action": row.get("agent_action"),
            "cot_source_action": row.get("agent_action"),
            "prompt_text": row.get("prompt_text"),
            "analysis_prefix_text": row.get("analysis_prefix_text"),
            "analysis_prefix_token_ids_json": row.get("analysis_prefix_token_ids_json"),
            "prefix_token_index": row.get("prefix_token_index"),
            "num_analysis_tokens": row.get("num_analysis_tokens"),
            "progress": row.get("progress"),
            "mismatch_type": "matched_prompt_cot_subset",
        })
        matched_rows.append(matched)
    return pd.DataFrame(mismatch_rows), pd.DataFrame(matched_rows)

mismatch_prefix_df = pd.DataFrame()
matched_prefix_df = pd.DataFrame()
mismatch_prefix_source = None

if FOLLOWUP_MISMATCH_PREFIX_PATH.exists():
    mismatch_prefix_df = normalize_existing_mismatch(pd.read_parquet(FOLLOWUP_MISMATCH_PREFIX_PATH))
    mismatch_prefix_source = str(FOLLOWUP_MISMATCH_PREFIX_PATH)
elif DIAG_MISMATCH_PREFIX_PATH.exists():
    mismatch_prefix_df = normalize_existing_mismatch(pd.read_parquet(DIAG_MISMATCH_PREFIX_PATH))
    mismatch_prefix_source = str(DIAG_MISMATCH_PREFIX_PATH)
    mismatch_prefix_df.to_parquet(FOLLOWUP_MISMATCH_PREFIX_PATH, index=False)
    print(f"Loaded existing diagnostics mismatch file and saved follow-up copy to {FOLLOWUP_MISMATCH_PREFIX_PATH}")
else:
    mismatch_prefix_df, matched_prefix_df = build_mismatch_and_matched(prefixes_df)
    mismatch_prefix_source = "constructed from outputs/prefixes.parquet"
    if not mismatch_prefix_df.empty:
        mismatch_prefix_df.to_parquet(FOLLOWUP_MISMATCH_PREFIX_PATH, index=False)
        print(f"Created {FOLLOWUP_MISMATCH_PREFIX_PATH} with {len(mismatch_prefix_df)} rows")

# Always create/update matched subset from the current selected rows when possible.
if FOLLOWUP_MATCHED_PREFIX_PATH.exists():
    matched_prefix_df = pd.read_parquet(FOLLOWUP_MATCHED_PREFIX_PATH)
else:
    _, matched_prefix_df = build_mismatch_and_matched(prefixes_df)
    if not matched_prefix_df.empty:
        matched_prefix_df.to_parquet(FOLLOWUP_MATCHED_PREFIX_PATH, index=False)
        print(f"Created {FOLLOWUP_MATCHED_PREFIX_PATH} with {len(matched_prefix_df)} rows")

mismatch_args = [
    "python", "scripts/03_score_prefix_actions_together.py",
    "--input", str(FOLLOWUP_MISMATCH_PREFIX_PATH),
    "--output", str(FOLLOWUP_MISMATCH_SCORE_PATH),
    "--model_name", "openai/gpt-oss-20b",
    "--api_key_env", "TOGETHER_API_KEY",
    "--scoring_method", "echo_teacher_forced",
    "--candidate_mode", "bare_action",
    "--temperature", "0.0",
    "--top_p", "1.0",
    "--logprobs", "20",
    "--max_tokens", "1",
    "--save_every", "50",
    "--resume",
]
matched_args = [
    "python", "scripts/03_score_prefix_actions_together.py",
    "--input", str(FOLLOWUP_MATCHED_PREFIX_PATH),
    "--output", str(FOLLOWUP_MATCHED_SCORE_PATH),
    "--model_name", "openai/gpt-oss-20b",
    "--api_key_env", "TOGETHER_API_KEY",
    "--scoring_method", "echo_teacher_forced",
    "--candidate_mode", "bare_action",
    "--temperature", "0.0",
    "--top_p", "1.0",
    "--logprobs", "20",
    "--max_tokens", "1",
    "--save_every", "50",
    "--resume",
]
mismatch_command_text = command_to_text(mismatch_args)
matched_command_text = command_to_text(matched_args)
print("Mismatched scoring command:")
print(mismatch_command_text)
print("\nMatched subset scoring command:")
print(matched_command_text)

if RUN_TOGETHER_SCORING:
    print("RUN_TOGETHER_SCORING=True; executing scoring commands.")
    subprocess.run(mismatch_args, check=True)
    subprocess.run(matched_args, check=True)
else:
    print("RUN_TOGETHER_SCORING=False; not executing API scoring commands.")

mismatch_scores = None
matched_scores = None
if FOLLOWUP_MISMATCH_SCORE_PATH.exists():
    mismatch_scores = normalize_existing_mismatch(pd.read_parquet(FOLLOWUP_MISMATCH_SCORE_PATH))
    mismatch_scores = recompute_prefix_metrics(mismatch_scores)
else:
    print(f"No mismatched scores found at {FOLLOWUP_MISMATCH_SCORE_PATH}")
if FOLLOWUP_MATCHED_SCORE_PATH.exists():
    matched_scores = recompute_prefix_metrics(ensure_decision_id(pd.read_parquet(FOLLOWUP_MATCHED_SCORE_PATH)))
else:
    print(f"No matched subset scores found at {FOLLOWUP_MATCHED_SCORE_PATH}")

mismatched_summary = pd.DataFrame()
if mismatch_scores is not None and set(Q_COLS + ["prompt_agent_action", "cot_agent_action"]).issubset(mismatch_scores.columns):
    mismatch_scores["commitment_to_prompt_action"] = q_for_action(mismatch_scores, "prompt_agent_action")
    mismatch_scores["commitment_to_cot_source_action"] = q_for_action(mismatch_scores, "cot_agent_action")
    if "top_action" not in mismatch_scores.columns:
        mismatch_scores["top_action"] = mismatch_scores[Q_COLS].astype(float).idxmax(axis=1).str.replace("q_", "", regex=False)
    mismatch_scores["follows_prompt"] = mismatch_scores["top_action"] == mismatch_scores["prompt_agent_action"]
    mismatch_scores["follows_cot"] = mismatch_scores["top_action"] == mismatch_scores["cot_agent_action"]
    mismatch_scores["prompt_minus_cot_commitment"] = mismatch_scores["commitment_to_prompt_action"] - mismatch_scores["commitment_to_cot_source_action"]
    mismatch_scores.to_parquet(OUT_DIR / "mismatched_prompt_cot_scored_rows.parquet", index=False)

    prompt_curve, _ = per_step_binned_curve(mismatch_scores.rename(columns={"prompt_source_decision_id": "decision_id"}), "commitment_to_prompt_action", n_bins=N_PROGRESS_BINS)
    cot_curve, _ = per_step_binned_curve(mismatch_scores.rename(columns={"prompt_source_decision_id": "decision_id"}), "commitment_to_cot_source_action", n_bins=N_PROGRESS_BINS)
    fig, ax = plt.subplots(figsize=(7.5, 5.0))
    for curve, label in [(prompt_curve, "prompt action"), (cot_curve, "CoT-source action")]:
        if not curve.empty:
            x = curve["progress_mid"].to_numpy(dtype=float)
            y = curve["mean"].to_numpy(dtype=float)
            sem = curve["sem"].fillna(0).to_numpy(dtype=float)
            ax.plot(x, y, marker="o", linewidth=2, label=label)
            ax.fill_between(x, y - sem, y + sem, alpha=0.15)
    ax.set_xlabel("progress")
    ax.set_ylabel("commitment")
    ax.set_ylim(0, 1)
    ax.set_title("Mismatched prompt vs CoT-source commitment")
    ax.legend()
    save_fig(fig, "mismatched_prompt_vs_cot_commitment_over_progress.png")

    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    vals = pd.Series({"follows_prompt": mismatch_scores["follows_prompt"].mean(), "follows_cot": mismatch_scores["follows_cot"].mean()})
    ax.bar(vals.index, vals.values, alpha=0.85)
    ax.set_ylim(0, 1)
    ax.set_ylabel("fraction of scored rows")
    ax.set_title("Mismatched top action: prompt vs CoT")
    save_fig(fig, "mismatched_top_action_follow_prompt_vs_cot.png")

    fig, ax = plt.subplots(figsize=(7.0, 4.8))
    ax.hist(mismatch_scores["prompt_minus_cot_commitment"].dropna(), bins=30, alpha=0.85)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("prompt minus CoT-source commitment")
    ax.set_ylabel("rows")
    ax.set_title("Prompt-minus-CoT commitment in mismatched control")
    save_fig(fig, "mismatched_prompt_minus_cot_histogram.png")

    fig, ax = plt.subplots(figsize=(6.0, 6.0))
    ax.scatter(mismatch_scores["commitment_to_prompt_action"], mismatch_scores["commitment_to_cot_source_action"], s=18, alpha=0.7)
    ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("commitment to prompt action")
    ax.set_ylabel("commitment to CoT-source action")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title("Mismatched prompt vs CoT-source action commitment")
    save_fig(fig, "mismatched_prompt_vs_cot_scatter.png")

    matched_mean = np.nan
    if matched_scores is not None and "commitment_to_final" in matched_scores.columns:
        matched_curve, _ = per_step_binned_curve(matched_scores, "commitment_to_final", n_bins=N_PROGRESS_BINS)
        fig, ax = plt.subplots(figsize=(7.5, 5.0))
        for curve, label in [(matched_curve, "matched commitment to final"), (prompt_curve, "mismatched commitment to prompt"), (cot_curve, "mismatched commitment to CoT-source")]:
            if curve is not None and not curve.empty:
                ax.plot(curve["progress_mid"], curve["mean"], marker="o", linewidth=2, label=label)
        ax.set_xlabel("progress")
        ax.set_ylabel("commitment")
        ax.set_ylim(0, 1)
        ax.set_title("Matched vs mismatched commitment curves")
        ax.legend()
        save_fig(fig, "matched_vs_mismatched_commitment_curves.png")
        matched_mean = matched_scores["commitment_to_final"].mean()

    mismatched_summary = pd.DataFrame([
        {"metric": "rows", "value": len(mismatch_scores)},
        {"metric": "mean_commitment_to_prompt_action", "value": mismatch_scores["commitment_to_prompt_action"].mean()},
        {"metric": "mean_commitment_to_cot_source_action", "value": mismatch_scores["commitment_to_cot_source_action"].mean()},
        {"metric": "fraction_top_action_follows_prompt", "value": mismatch_scores["follows_prompt"].mean()},
        {"metric": "fraction_top_action_follows_cot", "value": mismatch_scores["follows_cot"].mean()},
        {"metric": "mean_prompt_minus_cot_commitment", "value": mismatch_scores["prompt_minus_cot_commitment"].mean()},
        {"metric": "matched_subset_mean_commitment_to_final", "value": matched_mean},
    ])
else:
    mismatched_summary = pd.DataFrame([
        {"metric": "mismatched_prefix_rows_prepared", "value": len(mismatch_prefix_df)},
        {"metric": "matched_prefix_rows_prepared", "value": len(matched_prefix_df)},
        {"metric": "mismatched_scores_available", "value": False},
        {"metric": "run_together_scoring", "value": RUN_TOGETHER_SCORING},
    ])

mismatched_summary.to_csv(OUT_DIR / "mismatched_prompt_cot_summary.csv", index=False)
display(mismatched_summary)


## 5. Final Report

In [ ]:

def fmt(value, digits=3):
    if value is None:
        return "NA"
    try:
        if pd.isna(value):
            return "NA"
    except Exception:
        pass
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.{digits}f}"
    return str(value)

report = []
report.append("# Follow-Ups 1-4 Report")
report.append("")
report.append("## Dataset")
report.append(f"- Prefix points: {len(prefix_df):,}")
report.append(f"- Decision steps: {prefix_df['decision_id'].nunique():,}")
report.append(f"- Trajectories: {prefix_df['trajectory_id'].nunique() if 'trajectory_id' in prefix_df.columns else 'NA'}")
report.append("- This notebook does not expand to the full 100 trajectories.")

report.append("")
report.append("## 1. Permanent Lock-In")
report.append(f"- Median first lock-in: {fmt(permanent_lock_report.get('median_first_lock_in_progress'))}")
report.append(f"- Median permanent lock-in: {fmt(permanent_lock_report.get('median_permanent_lock_in_progress'))}")
report.append(f"- Fraction with permanent lock-in: {fmt(permanent_lock_report.get('fraction_with_permanent_lock_in'))}")
report.append(f"- Mean switches after first lock-in: {fmt(permanent_lock_report.get('mean_switch_count_after_first_lock'))}")
report.append(f"- Early-first/late-permanent examples: {len(early_first_late_perm)} rows in `outputs/followups/early_first_late_permanent_lock_examples.csv`.")
report.append(f"- High switch-after-first-lock examples: {len(high_switch_after_first)} rows in `outputs/followups/high_switch_after_first_lock_examples.csv`.")

report.append("")
report.append("## 2. Action-Wise Curves")
if 'action_gain_table' in globals() and not action_gain_table.empty:
    fastest = action_gain_table.sort_values("curve_gain", ascending=False).iloc[0]
    slowest = action_gain_table.sort_values("curve_gain", ascending=True).iloc[0]
    report.append(f"- Fastest commitment gain: {fastest['agent_action']} ({fmt(fastest['curve_gain'])}).")
    report.append(f"- Slowest commitment gain: {slowest['agent_action']} ({fmt(slowest['curve_gain'])}).")
else:
    report.append("- Commitment gain by action could not be computed.")
if 'actionwise_summary' in globals() and not actionwise_summary.empty:
    left = actionwise_summary[actionwise_summary["agent_action"] == "LEFT"].iloc[0] if (actionwise_summary["agent_action"] == "LEFT").any() else None
    right = actionwise_summary[actionwise_summary["agent_action"] == "RIGHT"].iloc[0] if (actionwise_summary["agent_action"] == "RIGHT").any() else None
    if left is not None and right is not None:
        report.append(f"- LEFT vs RIGHT mean CoT length: {fmt(left.get('mean_num_analysis_tokens'))} vs {fmt(right.get('mean_num_analysis_tokens'))}.")
        report.append(f"- LEFT vs RIGHT prompt-only commitment: {fmt(left.get('mean_prompt_only_commitment_to_final'))} vs {fmt(right.get('mean_prompt_only_commitment_to_final'))}.")
    counts = actionwise_summary.set_index("agent_action")["n_decisions"].to_dict() if "n_decisions" in actionwise_summary.columns else {}
    report.append(f"- Action imbalance caveat: decision counts by action are {counts}.")

report.append("")
report.append("## 3. Strict Pre-Action Analysis")
if 'strict_pre_action_summary' in globals() and not strict_pre_action_summary.empty:
    def report_curve(subset, metric):
        rows = strict_pre_action_summary[(strict_pre_action_summary["subset"] == subset) & (strict_pre_action_summary["metric"] == metric)]
        if rows.empty:
            return None
        row = rows.iloc[0]
        return row["start_mean"], row["last_strict_bin_mean"], row["last_n_steps"]
    any_commit = report_curve("strict_before_any_action_mention", "commitment_to_final")
    final_commit = report_curve("strict_before_final_action_mention", "commitment_to_final")
    any_buf_commit = report_curve("strict_before_any_action_mention_buffered", "commitment_to_final")
    final_buf_commit = report_curve("strict_before_final_action_mention_buffered", "commitment_to_final")
    if any_commit:
        report.append(f"- Strict before any action mention commitment changes from {fmt(any_commit[0])} to {fmt(any_commit[1])}; last-bin decision count {fmt(any_commit[2])}.")
    if final_commit:
        report.append(f"- Strict before first final-action mention commitment changes from {fmt(final_commit[0])} to {fmt(final_commit[1])}; last-bin decision count {fmt(final_commit[2])}.")
    if any_buf_commit:
        report.append(f"- Buffered strict before any action mention commitment changes from {fmt(any_buf_commit[0])} to {fmt(any_buf_commit[1])}.")
    if final_buf_commit:
        report.append(f"- Buffered strict before first final-action mention commitment changes from {fmt(final_buf_commit[0])} to {fmt(final_buf_commit[1])}.")
    if 'strict_coverage_table' in globals() and not strict_coverage_table.empty:
        late_cov = strict_coverage_table[strict_coverage_table["progress_bin"] >= N_PROGRESS_BINS - 3]
        min_late_fraction = late_cov["decision_fraction"].min() if not late_cov.empty else np.nan
        report.append(f"- Late-bin coverage caveat: minimum decision fraction in the last three strict normalized bins is {fmt(min_late_fraction)}.")
else:
    report.append("- Strict pre-action analysis was skipped because mention indices were unavailable.")

report.append("")
report.append("## 4. Mismatched Prompt/CoT Control")
report.append(f"- Mismatched prefix rows prepared: {len(mismatch_prefix_df)} from {mismatch_prefix_source}.")
report.append(f"- Matched subset prefix rows prepared: {len(matched_prefix_df)}.")
if mismatch_scores is not None and not mismatched_summary.empty and "follows_prompt" in mismatch_scores.columns:
    follows_prompt = mismatch_scores["follows_prompt"].mean()
    follows_cot = mismatch_scores["follows_cot"].mean()
    report.append(f"- Scored mismatched rows: {len(mismatch_scores)}.")
    report.append(f"- Top action follows prompt vs CoT-source: {fmt(follows_prompt)} vs {fmt(follows_cot)}.")
    if follows_prompt > follows_cot:
        report.append("- In this scored control, top actions follow the prompt/grid action more often than the CoT-source action.")
    elif follows_cot > follows_prompt:
        report.append("- In this scored control, top actions follow the CoT-source action more often than the prompt/grid action.")
    else:
        report.append("- In this scored control, neither prompt nor CoT-source dominates by top-action frequency.")
else:
    report.append("- Mismatched scores are not available. The notebook did not run Together AI because RUN_TOGETHER_SCORING is False.")
    report.append("- Exact command to score mismatched prefixes:")
    report.append("```bash")
    report.append(mismatch_command_text)
    report.append("```")
    report.append("- Exact command to score the matched subset:")
    report.append("```bash")
    report.append(matched_command_text)
    report.append("```")

report.append("")
report.append("## Updated Cautious Interpretation")
report.append("- The prompt induces an initial action prior.")
report.append("- Generated reasoning sharpens and stabilizes this prior into high action commitment.")
report.append("- Strict pre-action analyses test whether commitment rises before explicit action verbalization.")
report.append("- Mismatched prompt/CoT controls probe whether the behavioral commitment follows grid context or generated reasoning text.")
report.append("- Do not claim that CoT causally causes the action unless mismatched and additional intervention controls support it.")

if WARNINGS:
    report.append("")
    report.append("## Warnings")
    for msg in WARNINGS:
        report.append(f"- {msg}")

report_text = "\n".join(report) + "\n"
report_path = OUT_DIR / "followups_1_to_4_report.md"
report_path.write_text(report_text)
print(report_text)
print(f"Wrote {report_path}")
